# Lab 1: Flight Data Streaming Analysis

## Requirements:
- Read CSV files from streaming directory
- Aggregate by destination and origin countries  
- Show total count for each pair
- Display only changed records to console

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import os
import shutil

print("✓ Libraries imported")

✓ Libraries imported


In [5]:
spark = SparkSession.builder \
    .appName("FlightDataLab1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✓ Spark session ready")

✓ Spark session ready


In [6]:
schema = StructType([
    StructField("DEST_COUNTRY_NAME", StringType(), True),
    StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
    StructField("count", IntegerType(), True)
])

print("✓ Schema defined")

✓ Schema defined


In [7]:
input_path = "/home/jovyan/lab1/streaming_input"
data_path = "/home/jovyan/lab1/data"

os.makedirs(input_path, exist_ok=True)
os.makedirs(data_path, exist_ok=True)

# Clear input directory
for f in os.listdir(input_path):
    if f.endswith('.csv'):
        os.remove(os.path.join(input_path, f))

print(f"✓ Watching: {input_path}")
print(f"✓ Data source: {data_path}")

✓ Watching: /home/jovyan/lab1/streaming_input
✓ Data source: /home/jovyan/lab1/data


In [8]:
def process_batch(df, epoch_id):
    """Process each batch and show aggregated results"""
    result = df.groupBy("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME") \
                .agg(sum("count").alias("total_count"))
    
    if result.count() > 0:
        print(f"\n{'='*60}")
        print(f"Batch: {epoch_id} - Processing Time: {epoch_id * 5} seconds")
        print(f"{'='*60}")
        result.show(100, truncate=False)
        print(f"{'='*60}\n")

print("✓ Batch processor defined")

✓ Batch processor defined


In [9]:
stream_df = spark.readStream \
    .schema(schema) \
    .option("header", "true") \
    .option("maxFilesPerTrigger", 1) \
    .csv(input_path)

query = stream_df.writeStream \
    .foreachBatch(process_batch) \
    .trigger(processingTime="5 seconds") \
    .start()

print("✓ Streaming query started")
print(f"  Query ID: {query.id}")
print(f"  Status: Active")

✓ Streaming query started
  Query ID: c5cf4cf9-af0f-42b4-8570-eefd7ce57286
  Status: Active


In [ ]:
def add_file(filename):
    """Add a CSV file to streaming input directory"""
    source = os.path.join(data_path, filename)
    dest = os.path.join(input_path, filename)
    
    if os.path.exists(source):
        shutil.copy2(source, dest)
        print(f"✓ Added: {filename}")
        return True
    else:
        print(f"✗ Not found: {filename}")
        return False

def list_files():
    """List all available CSV files"""
    files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
    print("\n📁 Available CSV files:")
    for f in sorted(files):
        print(f"  • {f}")
    return files

def check_status():
    """Check streaming query status"""
    print(f"\n📊 Streaming Status:")
    print(f"  Active: {query.isActive}")
    print(f"  Query ID: {query.id}")
    
    files = [f for f in os.listdir(input_path) if f.endswith('.csv')]
    print(f"  Files in streaming_input: {len(files)}")
    for f in files:
        print(f"    - {f}")

print("✓ Helper functions ready")

✓ Helper functions ready


In [11]:
print("="*60)
print("🚀 Flight Data Streaming Application")
print("="*60)

list_files()
check_status()

print("\n💡 Commands:")
print("  • add_file('filename.csv')  - Add a file to process")
print("  • check_status()            - Check streaming status")
print("  • list_files()              - List available files")
print("="*60)

🚀 Flight Data Streaming Application

📁 Available CSV files:
  • 2010-summary.csv
  • 2011-summary.csv
  • 2012-summary.csv
  • 2013-summary.csv
  • 2014-summary.csv
  • 2015-summary.csv

📊 Streaming Status:
  Active: True
  Query ID: c5cf4cf9-af0f-42b4-8570-eefd7ce57286
  Files in streaming_input: 0

💡 Commands:
  • add_file('filename.csv')  - Add a file to process
  • check_status()            - Check streaming status
  • list_files()              - List available files


In [12]:
# Add your first file here
add_file('2010-summary.csv')

✓ Added: 2010-summary.csv


True


Batch: 0 - Processing Time: 0 seconds
+------------------------------+--------------------------------+-----------+
|DEST_COUNTRY_NAME             |ORIGIN_COUNTRY_NAME             |total_count|
+------------------------------+--------------------------------+-----------+
|Ireland                       |United States                   |231        |
|United States                 |Egypt                           |25         |
|India                         |United States                   |66         |
|Singapore                     |United States                   |25         |
|Grenada                       |United States                   |65         |
|United States                 |Costa Rica                      |501        |
|United States                 |Senegal                         |46         |
|Marshall Islands              |United States                   |77         |
|Sint Maarten                  |United States                   |61         |
|United States           

In [13]:
add_file('2011-summary.csv')

✓ Added: 2011-summary.csv


True


Batch: 1 - Processing Time: 5 seconds
+------------------------------+--------------------------------+-----------+
|DEST_COUNTRY_NAME             |ORIGIN_COUNTRY_NAME             |total_count|
+------------------------------+--------------------------------+-----------+
|Guinea                        |United States                   |5          |
|Saint Martin                  |United States                   |1          |
|Croatia                       |United States                   |2          |
|Romania                       |United States                   |4          |
|Ireland                       |United States                   |250        |
|United States                 |Egypt                           |15         |
|India                         |United States                   |73         |
|Singapore                     |United States                   |27         |
|Grenada                       |United States                   |67         |
|United States           

In [14]:
add_file('2012-summary.csv')
add_file('2013-summary.csv')
add_file('2014-summary.csv')
add_file('2015-summary.csv')

✓ Added: 2012-summary.csv
✓ Added: 2013-summary.csv
✓ Added: 2014-summary.csv
✓ Added: 2015-summary.csv


True


Batch: 2 - Processing Time: 10 seconds
+------------------------------+--------------------------------+-----------+
|DEST_COUNTRY_NAME             |ORIGIN_COUNTRY_NAME             |total_count|
+------------------------------+--------------------------------+-----------+
|Romania                       |United States                   |14         |
|Ireland                       |United States                   |255        |
|United States                 |Egypt                           |12         |
|India                         |United States                   |61         |
|Niger                         |United States                   |1          |
|Singapore                     |United States                   |21         |
|Grenada                       |United States                   |42         |
|United States                 |Costa Rica                      |549        |
|United States                 |Senegal                         |32         |
|Marshall Islands       

In [15]:
if query.isActive:
    query.stop()
    print("✓ Streaming stopped")
else:
    print("Streaming already stopped")

✓ Streaming stopped


In [16]:
# Read all processed files and show final aggregation
from pyspark.sql.functions import sum as _sum

all_files_df = spark.read.option("header", "true").csv(data_path + "/*.csv")

final_result = all_files_df \
    .groupBy("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME") \
    .agg(_sum("count").alias("total_count")) \
    .orderBy("total_count", ascending=False)

print("\n🏁 FINAL RESULT - All Years Combined:")
print("="*60)
final_result.show(50, truncate=False)


🏁 FINAL RESULT - All Years Combined:
+------------------+-------------------+-----------+
|DEST_COUNTRY_NAME |ORIGIN_COUNTRY_NAME|total_count|
+------------------+-------------------+-----------+
|United States     |United States      |2119795.0  |
|United States     |Canada             |49695.0    |
|Canada            |United States      |49052.0    |
|United States     |Mexico             |38225.0    |
|Mexico            |United States      |38075.0    |
|United Kingdom    |United States      |10946.0    |
|United States     |United Kingdom     |10358.0    |
|Japan             |United States      |9205.0     |
|United States     |Japan              |8643.0     |
|Germany           |United States      |8501.0     |
|United States     |Germany            |8380.0     |
|United States     |Dominican Republic |7194.0     |
|Dominican Republic|United States      |6858.0     |
|United States     |The Bahamas        |5775.0     |
|Brazil            |United States      |5635.0     |
|The Bah